<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/orientation-hero.svg" align="center" width="80%">
</div>

<br>

# ORIENTATION: START HERE

<br>

**About:** A guided entry point for the Tokenization: Text to Numbers repo - what is taught, how to verify your environment, and how to run your first real tokenizer before opening any lesson notebook.

**Learning Goals:**
- Describe the arc of the two lesson notebooks, from a hand-built byte-pair encoder to three production tokenizers
- Verify that tiktoken, tokenizers, sentencepiece and matplotlib are importable in your environment
- Encode and decode one sentence with a production tokenizer and read the pieces and ids it produces

**Keywords:** tokenization, byte-pair encoding, bpe, tiktoken, sentencepiece, orientation, environment setup

**Prerequisite Knowledge:** (1) Basic Python, (2) Familiarity with Jupyter notebooks

**Target User:** Anyone starting the Tokenization: Text to Numbers curriculum

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: WHAT THIS REPO TEACHES](#Part_1)
> #### [PART 2: VERIFYING YOUR ENVIRONMENT](#Part_2)
> #### [PART 3: YOUR FIRST CALL](#Part_3)

<br>

In [ ]:
# Setup - all imports for this notebook; run this cell first
import sys

# Dependencies are declared in pyproject.toml and pinned by uv.lock.
# A missing package is recorded as None so Part 2 can report it instead of crashing here.
try:
    import tiktoken
except ImportError:
    tiktoken = None

try:
    import tokenizers
except ImportError:
    tokenizers = None

try:
    import sentencepiece
except ImportError:
    sentencepiece = None

try:
    import matplotlib
except ImportError:
    matplotlib = None

print(f"Python: {sys.version.split()[0]}")

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **WHAT** this repo **TEACHES**

A language model never reads letters or words - it reads a sequence of integers, and the tokenizer is the table that turns your text into those integers. This repo teaches that table from both ends. In the first notebook you build a byte-pair-encoding (BPE) tokenizer by hand on a tiny corpus, so you see exactly how repeated merging of the most frequent adjacent pair produces a vocabulary, and how that vocabulary trades off against sequence length. In the second notebook you run the same text through three production tokenizers (tiktoken, Hugging Face `tokenizers`, and SentencePiece), compare what each one produces, and learn which workloads - code, numbers, names, non-English text - cost far more tokens than their word count suggests.

The reason this matters for a practitioner is that context limits, price and latency are all counted in tokens, not words. A per-token price means nothing until you know which tokenizer produced the count.

___

**Note:** No API keys are needed anywhere in this repo. Every notebook runs locally with the packages from `pyproject.toml`. The only network access is a one-time download of the `cl100k_base` vocabulary file by tiktoken (Part 3 below), and a one-time download of the GPT-2 vocabulary by `tokenizers` in the second lesson.

___

#### COVERAGE MAP

| Notebook | What you can do after it |
|---|---|
| `01_bpe_by_hand.ipynb` (about 45 min) | Build a BPE vocabulary by counting and merging pairs, encode and decode a new sentence with it, and measure how vocabulary size trades off against sequence length |
| `02_three_real_tokenizers.ipynb` (about 45 min) | Compare tiktoken, Hugging Face `tokenizers` and SentencePiece on the same text, explain why their counts differ, and decide what to measure before trusting a per-token price |

Work through them in order: the second notebook assumes the hand-built encoder from the first.

**After completing Part 1 you should be able to:**

- Recognize that a model's input is a sequence of token ids and that context, price and latency are all counted in tokens
- Identify which notebook to open for hand-built BPE versus production tokenizer comparison
- Differentiate a tokenizer (text to ids) from the model that consumes the ids

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **A teammate reports that their monthly bill rose sharply after the same prompts were translated from English to Spanish, with no change in word count. Name the notebook that teaches how to measure this, and write the first measurement you would take.**

<br>

In [ ]:
# Write your answer as Python strings
notebook = ""  # the filename of the lesson notebook
first_measurement = ""  # one sentence: what you would count, and with which tool

print(notebook)
print(first_measurement)

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): `02_three_real_tokenizers.ipynb`. Count tokens for matched English and Spanish samples with the tokenizer of the model being billed. Word count is the wrong unit because accented and non-English text splits into more pieces per word, so the bill follows token count, not words. -->

<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **VERIFYING** your **ENVIRONMENT**

Before you open a lesson notebook, confirm that the four packages it uses are importable. A missing package otherwise surfaces as a stack trace in the middle of a lesson cell. The four packages are:

- `tiktoken` - the production tokenizer library used by OpenAI models
- `tokenizers` - the Hugging Face tokenizers library (byte-level BPE such as GPT-2)
- `sentencepiece` - a tokenizer library that trains and runs its own subword model
- `matplotlib` - used for the plots saved to `images/`

___

**Note:** No API keys are needed anywhere in this repo. There is no `.env` file to create and no secret to set. If you are about to paste a key into a cell, stop - nothing here uses one.

___

In [ ]:
# Environment verification - run this cell and read the output
# Versions are read from each package at run time, so nothing is hard-coded here.
# Declared minimums live in pyproject.toml at the repo root.

packages = {
    "tiktoken": tiktoken,
    "tokenizers": tokenizers,
    "sentencepiece": sentencepiece,
    "matplotlib": matplotlib,
}

missing = []
for name, module in packages.items():
    if module is None:
        print(f"[FAIL] {name} is NOT importable")
        missing.append(name)
    else:
        print(f"[OK]   {name} {getattr(module, '__version__', 'version unknown')}")

print()
print("No API keys are needed anywhere in this repo.")
print()

if not missing:
    print("All checks passed. You are ready for Part 3.")
else:
    print(f"Missing: {', '.join(missing)}")
    print("Install the project environment from the repo root with:")
    print("    uv sync")
    print("Then restart the kernel and re-run the Setup cell above.")

<br>

**If any package check failed:**

Run this from the repository root in a terminal:

```bash
uv sync
```

`pyproject.toml` declares what this repo imports and `uv.lock` pins the exact resolved versions, so every reader gets the same environment. If you use the Nix dev shell (`nix develop`), the environment is prepared for you on entry; start Jupyter with `uv run jupyter lab`.

Then restart the kernel and re-run the Setup cell at the top of this notebook.

**After completing Part 2 you should be able to:**

- Recognize which of the four packages is missing from an `[FAIL]` line
- Identify the one command that repairs the environment (`uv sync`) and where to run it
- Explain why this repo needs no API key or `.env` file

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Implement `version_tuple` so it turns a version string such as `"3.9.2"` into the tuple `(3, 9, 2)`, then use it to test whether your installed matplotlib meets the minimum declared in `pyproject.toml`. Why is comparing the raw strings the wrong approach?**

<br>

In [ ]:
def version_tuple(version: str) -> tuple:
    # TODO: write your implementation here
    return ()


minimum = version_tuple("3.9.0")
installed = version_tuple(matplotlib.__version__) if matplotlib is not None else ()
print(installed, minimum)

# Then explain, as a comment below, why comparing the raw strings fails
# Why strings fail: ...

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): `tuple(int(part) for part in version.split(".")[:3])` for plain release strings (pre-release suffixes such as 3.10.0rc1 would need the `packaging` library). Raw strings compare character by character, so "3.10.0" < "3.9.0" because "1" < "9"; tuples of ints compare numerically, so (3, 10, 0) > (3, 9, 0). -->

<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **YOUR FIRST** call

Before the lessons build tokenizers from scratch, it helps to see what a finished one produces. The cell below encodes one sentence with tiktoken's `cl100k_base` encoding, prints the text piece behind every id, and decodes the ids back to the original string.

Three calls do all the work: `tiktoken.get_encoding` loads a named encoding, `encode` turns text into a list of integer ids, and `decode` turns ids back into text. The piece list is built by decoding each id on its own, so you can see exactly where the tokenizer cut the sentence.

___

**Note:** `get_encoding` needs the `cl100k_base` vocabulary file, which tiktoken fetches over the network the first time you use it. If you are offline on the first run, the cell prints a fallback message instead of failing. Call pattern checked against the tiktoken README, retrieved 2026-10-16 - re-check at [github.com/openai/tiktoken](https://github.com/openai/tiktoken).

___

In [ ]:
# Verified against the tiktoken README (get_encoding, encode, decode), 2026-10-16
# Re-check at https://github.com/openai/tiktoken

SENTENCE = "Tokenization isn't magic."

enc = None
if tiktoken is None:
    print("tiktoken is not importable - fix the environment in Part 2 first.")
else:
    try:
        enc = tiktoken.get_encoding("cl100k_base")
    except Exception as err:
        # The vocabulary file is fetched on first use, so the usual cause is no network.
        print(f"Could not load cl100k_base ({type(err).__name__}).")
        print("Connect to the internet, re-run this cell once, and the file is cached for later runs.")

if enc is not None:
    ids = enc.encode(SENTENCE)
    # Decode each id alone to show the piece behind it; repr makes leading spaces visible
    pieces = [enc.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in ids]
    decoded = enc.decode(ids)

    print(f"Text:    {SENTENCE!r}")
    print(f"Pieces:  {pieces}")
    print(f"Ids:     {ids}")
    print(f"Decoded: {decoded!r}")
    print(f"Round trip exact: {decoded == SENTENCE}")

<br>

**What to notice:** look at where the spaces sit in the piece list. A space is not its own token - it travels with the word that comes after it, so `" magic"` is a different piece (with a different id) from `"magic"`. You will see why in the first lesson, when you build the merge table yourself.

**After completing Part 3 you should be able to:**

- Recognize that `encode` returns integer ids and `decode` is an exact inverse
- Identify the leading-space convention in a token listing
- Differentiate the text pieces you read from the ids the model receives

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Encode the word `magic` with and without a leading space, print both id lists, and state in a comment what the difference tells you about how this tokenizer treats whitespace.**

<br>

In [ ]:
bare = "magic"
spaced = " magic"

if enc is not None:
    # TODO: print the ids for bare and for spaced
    pass

# What the difference tells you: ...

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): `print(enc.encode(bare), enc.encode(spaced))`. The two lists differ, because the vocabulary holds the space-prefixed word as its own entry; the tokenizer does not strip whitespace and then re-add it. The practical consequence is that the same word costs a different id at the start of a string than mid-sentence, which is why token counts depend on exact text and not only on words. -->

<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

## **Sources** Consulted

- [tiktoken README](https://github.com/openai/tiktoken) - retrieved 2026-10-16

<hr style="border: 6px solid#003262;" />